# Shadow Analysis Demo — SolarSense AR

Uses the `ephem` library to compute sun path for any Indian city.
- Calculates hourly sun azimuth + altitude across 12 months
- Visualises solar availability heat map
- Computes monthly shading loss factor


In [ ]:
# !pip install ephem matplotlib numpy

In [ ]:
import ephem
import numpy as np
import matplotlib.pyplot as plt
import datetime

# Indian cities
CITIES = {
    'Nagpur':    ('21.15', '79.09'),
    'Jaipur':    ('26.91', '75.79'),
    'Mumbai':    ('19.07', '72.87'),
    'Chennai':   ('13.08', '80.27'),
    'Kolkata':   ('22.57', '88.37'),
}
print('Cities configured:', list(CITIES.keys()))

In [ ]:
def compute_sun_path(lat: str, lng: str, year: int = 2025):
    """Return dict of {month: list of (hour, altitude_deg, azimuth_deg)}"""
    observer = ephem.Observer()
    observer.lat = lat
    observer.lon = lng
    observer.pressure = 0  # ignore atmospheric refraction

    sun = ephem.Sun()
    result = {}

    for month in range(1, 13):
        day = 15  # mid-month representative day
        hourly = []
        for hour in range(5, 20):  # 5am to 7pm
            observer.date = datetime.datetime(year, month, day, hour - 5, 30)
            sun.compute(observer)
            alt_deg = float(sun.alt) * 180 / 3.14159265
            az_deg  = float(sun.az)  * 180 / 3.14159265
            if alt_deg > 0:
                hourly.append((hour, alt_deg, az_deg))
        result[month] = hourly

    return result

# Run for Nagpur
nagpur_path = compute_sun_path(*CITIES['Nagpur'])
print(f'Nagpur Dec noon altitude: {nagpur_path[12][len(nagpur_path[12])//2][1]:.1f}°')
print(f'Nagpur Jun noon altitude: {nagpur_path[6][len(nagpur_path[6])//2][1]:.1f}°')

In [ ]:
# ── Sun altitude heat map (month vs hour) ─────────────────────────────────────
months = list(range(1, 13))
hours = list(range(5, 20))
MONTH_NAMES = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

heat = np.zeros((12, 15))
for m_idx, month in enumerate(months):
    for entry in nagpur_path[month]:
        h_idx = entry[0] - 5
        heat[m_idx, h_idx] = entry[1]

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(heat, aspect='auto', cmap='YlOrRd', origin='upper')
ax.set_xticks(range(15))
ax.set_xticklabels([f'{h}:00' for h in hours], rotation=45)
ax.set_yticks(range(12))
ax.set_yticklabels(MONTH_NAMES)
ax.set_title('Sun Altitude (°) — Nagpur | Heat Map (month × hour)', fontsize=13)
plt.colorbar(im, ax=ax, label='Solar Altitude (degrees)')
plt.tight_layout()
plt.savefig('nagpur_sun_path.png', dpi=150)
plt.show()

In [ ]:
# ── Monthly daylight hours & effective irradiance factor ─────────────────────
def monthly_effective_hours(sun_path: dict, min_altitude: float = 10.0) -> dict:
    """Return effective generation hours per month (alt > min_altitude)."""
    return {
        month: sum(1 for h, alt, az in entries if alt > min_altitude)
        for month, entries in sun_path.items()
    }

effective_hours = monthly_effective_hours(nagpur_path)

fig, ax = plt.subplots(figsize=(10, 4))
hours_vals = [effective_hours[m] for m in range(1, 13)]
bars = ax.bar(MONTH_NAMES, hours_vals, color='#F5A623', edgecolor='white')
ax.set_ylabel('Effective Generation Hours/Day')
ax.set_title('Nagpur — Monthly Effective Solar Hours (altitude > 10°)')
for bar, h in zip(bars, hours_vals):
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.1, str(h), ha='center', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ── Shadow loss estimation from obstacle height ───────────────────────────────
def shadow_loss_from_obstacle(
    obstacle_height_m: float,
    distance_m: float,
    sun_altitude_deg: float
) -> float:
    """Return fraction of panel area in shadow."""
    import math
    shadow_length = obstacle_height_m / math.tan(math.radians(max(sun_altitude_deg, 1)))
    if shadow_length <= distance_m:
        return 0.0  # no shadow reaches panel
    overlap = min(shadow_length - distance_m, 1.75)  # panel height ~1.75m
    return overlap / 1.75

# Example: AC unit 1m tall, 2m from panels, at noon in December (alt ~45°)
loss = shadow_loss_from_obstacle(obstacle_height_m=1.0, distance_m=2.0, sun_altitude_deg=45)
print(f'Shadow loss from AC unit: {loss*100:.1f}%')

# At sunrise (alt ~15°)
loss_sunrise = shadow_loss_from_obstacle(obstacle_height_m=1.0, distance_m=2.0, sun_altitude_deg=15)
print(f'Shadow loss at sunrise/sunset: {loss_sunrise*100:.1f}%')

In [ ]:
# ── Annual shadow loss for all cities ─────────────────────────────────────────
for city, coords in CITIES.items():
    path = compute_sun_path(*coords)
    eff = monthly_effective_hours(path)
    annual_hrs = sum(eff.values())
    print(f'{city:12s}: {annual_hrs} effective solar hours/year')